# Week 4: Transfer learning — guided seminar

This notebook is an ungraded, complete example.
We adapt a pretrained BERT encoder to named entity recognition (NER).
The homework uses the same tools on a different task: duplicate-question detection.
You may reuse this seminar's code in your homework.
Submit only the homework, with its experiment report; there is no seminar submission.

By the end, explain where pretrained weights enter the model,
how word labels become token labels, and which parameters receive gradients.
The final short example connects this workflow to T5's pretraining objective.

In [ ]:
# Colab already provides PyTorch, Transformers 5 and Datasets.
# Elsewhere, install PyTorch for your CPU/GPU, then transformers>=5.18,<6 and datasets>=3,<5.
import gc
import time
from collections import Counter

import datasets
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    pipeline,
    set_seed,
)

SEED = 42
BASE_MODEL = "google-bert/bert-base-cased"
MODEL_REVISION = "cd5ef92a9fb2f889e972770a36d4ed042daf221e"
DATASET_REVISION = "19edcb426bfd625c275c17b9a99b2239243f4377"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
print("Versions:", torch.__version__, transformers.__version__, datasets.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))

## 1. Tokens, padding and attention masks

BERT uses subword tokens, with `[CLS]` and `[SEP]` as special tokens.
A word can occupy several token positions.
Dynamic padding adds padding only up to the longest sequence in the current batch.
`attention_mask` distinguishes real input positions from padding;
it is not the supervision mask used in the loss.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True
)
batch = tokenizer(
    ["Michael studies transfer learning.", "An electroencephalographic recording."],
    padding=True,
    return_tensors="pt",
)
print({key: tuple(value.shape) for key, value in batch.items()})
for row, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(list(zip(tokenizer.convert_ids_to_tokens(row.tolist()), mask.tolist())))

In [ ]:
pair = tokenizer("Can an encoder learn useful features?", "Yes, from pretraining.")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print("Segments:", pair["token_type_ids"])

## 2. A pretrained encoder and a task head

The base encoder returns a contextual vector at every token position:
`[batch, sequence, hidden]`.
The masked-language-model head maps each vector to vocabulary logits:
`[batch, sequence, vocabulary]`.
A NER head will instead predict an entity label at each token position.
Loading an encoder's weights does not supply a trained head for every downstream task.

Transformers prints a load report for the checkpoint.
The pooler and the next-sentence-prediction head are `UNEXPECTED`:
the checkpoint keeps them from BERT pretraining, but the MLM model does not use them.

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained(BASE_MODEL, revision=MODEL_REVISION)
mlm.eval()
with torch.no_grad():
    representations = mlm.base_model(**batch).last_hidden_state
    vocabulary_logits = mlm(**batch).logits
print("Encoder output:", tuple(representations.shape))
print("MLM logits:", tuple(vocabulary_logits.shape))

Predict a missing word in two contexts.
Discuss how the distribution changes; this is an illustration of the objective,
not a factual-knowledge quiz or a method of evaluating the model's explanations.

In [ ]:
fill_mask = pipeline("fill-mask", model=mlm, tokenizer=tokenizer, device=-1)
for text in ["The doctor examined the [MASK].", "The mechanic examined the [MASK]."]:
    print(text)
    print([(item["token_str"], round(item["score"], 3)) for item in fill_mask(text)])

In [ ]:
del fill_mask, mlm, representations, vocabulary_logits
gc.collect()

## 3. NER data: words are not model tokens

CoNLL-2003 annotates words with IOB2 entity tags.
We use a fixed small subset to demonstrate the full workflow.
These sizes and the epoch count are teaching defaults, not a measured runtime budget.
Use a GPU if available; CPU execution of BERT can be slow.
The official test split is left untouched.

In [ ]:
raw = datasets.load_dataset("lhoestq/conll2003", revision=DATASET_REVISION)
label_names = raw["train"].features["ner_tags"].feature.names
id2label = dict(enumerate(label_names))
label2id = {label: index for index, label in id2label.items()}
train_words = raw["train"].shuffle(seed=SEED).select(range(2048))
dev_words = raw["validation"].shuffle(seed=SEED).select(range(512))
print("Labels:", label_names)
print(list(zip(train_words[0]["tokens"], train_words[0]["ner_tags"])))

We supervise only the first subword of each word.
Other subwords, special tokens and padding receive label `-100`.
PyTorch cross-entropy ignores those label positions.
They can still be visible to the encoder: ignoring a loss is different from masking attention.
Another valid policy labels all subwords, but requires handling B-/I- tags consistently.

In [ ]:
def align_word_labels(word_ids, word_labels):
    aligned = []
    previous_word = None
    for word_id in word_ids:
        if word_id is None or word_id == previous_word:
            aligned.append(-100)
        else:
            aligned.append(word_labels[word_id])
        previous_word = word_id
    return aligned


def tokenize_and_align(samples):
    encoded = tokenizer(
        samples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
    )
    encoded["labels"] = [
        align_word_labels(encoded.word_ids(batch_index=i), labels)
        for i, labels in enumerate(samples["ner_tags"])
    ]
    return encoded


assert align_word_labels([None, 0, 1, 1, 2, None], [1, 0, 5]) == [-100, 1, 0, -100, 5, -100]
example = tokenize_and_align(train_words[:1])
print(list(zip(
    tokenizer.convert_ids_to_tokens(example["input_ids"][0]),
    example["labels"][0],
)))

In [ ]:
train_tokens = train_words.map(
    tokenize_and_align, batched=True, remove_columns=train_words.column_names
)
dev_tokens = dev_words.map(
    tokenize_and_align, batched=True, remove_columns=dev_words.column_names
)
collator = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100)
train_loader = DataLoader(
    train_tokens, batch_size=16, shuffle=True, collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
dev_loader = DataLoader(dev_tokens, batch_size=16, shuffle=False, collate_fn=collator)
sample_batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in sample_batch.items()})
print("Attention mask:", sample_batch["attention_mask"][0].tolist())
print("Loss labels:", sample_batch["labels"][0].tolist())

## 4. Which parameters learn?

BERT is pretrained; the nine-label token classifier is randomly initialized.
In the load report, the classifier is `MISSING`: it is newly initialized.
Pretraining heads this model does not use are `UNEXPECTED`.
First freeze the backbone and inspect gradients from one batch.
There is no optimizer step in this demonstration, so it does not change the weights.
`requires_grad=False` freezes parameters; `eval()` disables dropout.
Neither operation eliminates the encoder forward pass.

In [ ]:
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    revision=MODEL_REVISION,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)


def parameter_counts(model):
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(
            parameter.numel() for parameter in model.parameters() if parameter.requires_grad
        ),
    }


for parameter in model.base_model.parameters():
    parameter.requires_grad_(False)
model.train()
model.base_model.eval()
device_batch = {key: value.to(DEVICE) for key, value in sample_batch.items()}
model(**device_batch).loss.backward()
print("Frozen:", parameter_counts(model))
print("Encoder has gradients:", any(p.grad is not None for p in model.base_model.parameters()))
print("Head has gradients:", any(p.grad is not None for p in model.classifier.parameters()))
assert all(p.grad is None for p in model.base_model.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())
model.zero_grad(set_to_none=True)
for parameter in model.base_model.parameters():
    parameter.requires_grad_(True)
print("Full fine-tuning:", parameter_counts(model))

## 5. One complete fine-tuning run

Accuracy over all tokens can hide NER errors because many words have tag `O`.
We use entity-level precision, recall and F1 with strict IOB2 matching.
An entity is `B-X` followed by any number of `I-X` tags.
A prediction counts only if its type and both boundaries match a gold entity.
An `I-X` tag that does not continue an entity of type X starts no entity.
Filter `-100` positions before computing metrics.
Use validation to inspect progress; never tune on the test set.
This small run demonstrates adaptation, not a claim about BERT's attainable NER quality.

In [ ]:
def entity_spans(tags):
    spans, start, kind = set(), None, None
    for position, tag in enumerate(tags + ["O"]):
        if start is not None and tag != f"I-{kind}":
            spans.add((kind, start, position))
            start = None
        if tag.startswith("B-"):
            start, kind = position, tag[2:]
    return spans


def entity_scores(references, predictions):
    gold_count = predicted_count = correct = 0
    for reference, prediction in zip(references, predictions):
        gold, predicted = entity_spans(reference), entity_spans(prediction)
        gold_count += len(gold)
        predicted_count += len(predicted)
        correct += len(gold & predicted)
    precision = correct / predicted_count if predicted_count else 0.0
    recall = correct / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


assert entity_spans(["B-PER", "I-PER", "O", "I-LOC", "B-ORG", "I-LOC"]) == {("PER", 0, 2), ("ORG", 4, 5)}
assert entity_scores([["B-PER", "I-PER", "B-LOC"]], [["B-PER", "O", "B-LOC"]]) == {
    "precision": 0.5, "recall": 0.5, "f1": 0.5,
}


@torch.no_grad()
def predict_word_tags(model, loader):
    model.eval()
    references, predictions = [], []
    for batch in loader:
        labels = batch["labels"]
        inputs = {key: value.to(DEVICE) for key, value in batch.items() if key != "labels"}
        predicted = model(**inputs).logits.argmax(dim=-1).cpu()
        for predicted_row, label_row in zip(predicted.tolist(), labels.tolist()):
            kept = [(prediction, label) for prediction, label in zip(predicted_row, label_row) if label != -100]
            predictions.append([id2label[prediction] for prediction, _ in kept])
            references.append([id2label[label] for _, label in kept])
    return references, predictions


def evaluate_ner(model, loader):
    return entity_scores(*predict_word_tags(model, loader))


print("Before training:", evaluate_ner(model, dev_loader))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
EPOCHS = 3
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
started = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    total_loss, batches = 0.0, 0
    for batch in train_loader:
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        loss = model(**inputs).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        batches += 1
    print({"epoch": epoch + 1, "train_loss": total_loss / batches, **evaluate_ner(model, dev_loader)})
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
print("Training + validation seconds:", time.perf_counter() - started)

At inference time there are no labels to mark the first subword of each word.
Use `word_ids()` to map token predictions back to words, then compare them with the gold tags.

In [ ]:
model.eval()
sample = dev_words[0]
encoded = tokenizer(sample["tokens"], is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
with torch.no_grad():
    predicted = model(**{key: value.to(DEVICE) for key, value in encoded.items()}).logits.argmax(-1)[0].cpu().tolist()
previous_word = None
for token_position, word_id in enumerate(encoded.word_ids()):
    if word_id is not None and word_id != previous_word:
        print(sample["tokens"][word_id], id2label[sample["ner_tags"][word_id]], id2label[predicted[token_position]])
    previous_word = word_id

One sentence rarely contains every kind of error, so classify errors over the whole dev subset.
For each gold entity without an exact match:
a predicted entity with the same boundaries but another type is a **wrong type**;
an overlapping prediction with different boundaries is a **boundary** error;
no overlapping prediction means the model **missed** the entity.
Predicted entities that overlap no gold entity are false positives; this cell does not list them.
Discuss one example of each category.
After a short run a category can still be absent; do not invent examples.

In [ ]:
def unmatched_gold_entities(reference, prediction):
    predicted = sorted(entity_spans(prediction), key=lambda span: span[1])
    for kind, start, end in sorted(entity_spans(reference) - set(predicted), key=lambda span: span[1]):
        same_boundaries = [span for span in predicted if span[1:] == (start, end)]
        overlapping = [span for span in predicted if span[1] < end and start < span[2]]
        if same_boundaries:
            yield "wrong type", (kind, start, end), same_boundaries
        elif overlapping:
            yield "boundary", (kind, start, end), overlapping
        else:
            yield "missed", (kind, start, end), []


def span_text(words, span):
    kind, start, end = span
    return f"{kind} '{' '.join(words[start:end])}'"


dev_references, dev_predictions = predict_word_tags(model, dev_loader)
counts, examples = Counter(), {}
for index, (reference, prediction) in enumerate(zip(dev_references, dev_predictions)):
    for category, gold, predicted in unmatched_gold_entities(reference, prediction):
        counts[category] += 1
        examples.setdefault(category, (index, gold, predicted))
print("Gold entities:", sum(len(entity_spans(reference)) for reference in dev_references))
print("Without an exact match:", {category: counts[category] for category in ("missed", "wrong type", "boundary")})
for category, (index, gold, predicted) in examples.items():
    words = dev_words[index]["tokens"]
    found = ", ".join(span_text(words, span) for span in predicted) or "nothing"
    print(f"\n{category}: gold {span_text(words, gold)}, predicted {found}")
    print("   ", " ".join(words))

## 6. A short connection to T5

BERT predicts masked positions through an encoder head.
T5 gives the encoder a corrupted sequence and trains the decoder to generate missing spans.
Each span has its own sentinel, and the target ends with another sentinel.
Both encoder input and target end with EOS.
This word-level toy example illustrates the sequence layout;
a real T5 tokenizer operates on subword IDs, not these space-separated words.
We do not load or train another model here.

In [ ]:
source_tokens = ["The", "small", "cat", "sat", "on", "the", "warm", "mat", "."]
encoder_tokens = ["The", "<extra_id_0>", "sat", "on", "the", "<extra_id_1>", ".", "</s>"]
target_tokens = ["<extra_id_0>", "small", "cat", "<extra_id_1>", "warm", "mat", "<extra_id_2>", "</s>"]
print("Original:", " ".join(source_tokens))
print("Encoder input:", " ".join(encoder_tokens))
print("Target:", " ".join(target_tokens))

Teacher forcing gives the decoder previous **gold** target tokens during training.
Its self-attention is causal; cross-attention can access the encoded input.
For T5, the decoder starts with the pad token, then receives the shifted target.
Loss covers target sentinels, missing words and EOS; padding labels are ignored.
The start token is a real decoder position even though its ID equals the pad ID.

In [ ]:
toy_vocabulary = {"<pad>": 0, "</s>": 1}
for token in encoder_tokens + target_tokens:
    if token not in toy_vocabulary:
        toy_vocabulary[token] = len(toy_vocabulary)
labels = [toy_vocabulary[token] for token in target_tokens] + [-100, -100]
decoder_input_ids = [toy_vocabulary["<pad>"]] + [
    token_id if token_id != -100 else toy_vocabulary["<pad>"]
    for token_id in labels[:-1]
]
print("Toy labels:", labels)
print("Shifted decoder inputs:", decoder_input_ids)
print("Positions contributing to loss:", [label != -100 for label in labels])
assert len(labels) == len(decoder_input_ids)

## Take this to the homework

Reuse tokenization, dynamic padding, device handling, gradient inspection and the training loop.
Replace token labels with one binary label per question pair.
Compare a frozen encoder plus a trained head with full fine-tuning,
starting both runs from the same pretrained encoder and the same initial head.
There is one homework submission worth 9 required points; the seminar is practice.

References: [HF token-classification guide](https://huggingface.co/docs/transformers/en/tasks/token_classification),
[BERT](https://arxiv.org/abs/1810.04805), [T5](https://jmlr.org/papers/v21/20-074.html).